In [ ]:
# ============================================================
# BLOCO 1 — CARGA E DIAGNÓSTICO INICIAL
# Telecom X BR II — Churn Prediction
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from urllib.request import urlopen

print("📊 TELECOM X BR II — CHURN PREDICTION")
print("=" * 60)

# ------------------------------------------------------------
# 1. CARREGAR DATASET TRATADO
# ------------------------------------------------------------

url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json"
with urlopen(url, timeout=30) as response:
    df = pd.json_normalize(json.load(response))

df["Churn"] = df["Churn"].str.strip().replace("", np.nan)
df["account.Charges.Total"] = pd.to_numeric(df["account.Charges.Total"], errors="coerce")

print("\n✅ Dataset carregado com sucesso!")
print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

# ------------------------------------------------------------
# 2. VISUALIZAÇÃO INICIAL
# ------------------------------------------------------------

print("\n🔎 Primeiras 5 linhas:")
display(df.head())

# ------------------------------------------------------------
# 3. TIPOS DAS VARIÁVEIS
# ------------------------------------------------------------

print("\n📋 Tipos das variáveis:")
display(df.dtypes.to_frame("dtype"))

# ------------------------------------------------------------
# 4. VALORES AUSENTES
# ------------------------------------------------------------

print("\n🔍 Valores ausentes:")
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing) == 0:
    print("✅ Nenhum valor ausente encontrado.")
else:
    display(missing.to_frame("missing"))

# ------------------------------------------------------------
# 5. DUPLICATAS
# ------------------------------------------------------------

duplicadas = df.duplicated().sum()

print(f"\n🔁 Linhas duplicadas: {duplicadas}")

# ------------------------------------------------------------
# 6. COLUNAS
# ------------------------------------------------------------

print("\n📌 Colunas disponíveis:")

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

# ------------------------------------------------------------
# 7. POSSÍVEIS VARIÁVEIS DE CHURN
# ------------------------------------------------------------

churn_cols = [col for col in df.columns if "churn" in col.lower()]

print("\n🎯 Colunas relacionadas a churn:")
print(churn_cols)

print("\n" + "=" * 60)
print("✅ BLOCO 1 CONCLUÍDO")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCO 2 — TARGET + PREPARAÇÃO DE X / y
# Telecom X BR II — Churn Prediction
# ============================================================

import pandas as pd
import numpy as np

print("🎯 BLOCO 2 — PREPARAÇÃO DO TARGET")
print("=" * 60)

# ------------------------------------------------------------
# 1. DEFINIR TARGET
# ------------------------------------------------------------

target_col = "Churn"

if target_col not in df.columns:
    raise ValueError("❌ Coluna 'Churn' não encontrada.")

print(f"\n✅ Target identificado: {target_col}")

# ------------------------------------------------------------
# 2. AUDITORIA INICIAL
# ------------------------------------------------------------

print("\n📊 Distribuição original:")
print(df[target_col].value_counts(dropna=False))

total_original = len(df)
missing_target = df[target_col].isna().sum()

print(f"\nTotal original: {total_original:,}")
print(f"Target ausente: {missing_target:,}")

# ------------------------------------------------------------
# 3. REMOVER REGISTROS SEM TARGET
# ------------------------------------------------------------

df_model = df.dropna(subset=[target_col]).copy()

removed = total_original - len(df_model)

print(f"\n🧹 Registros removidos por target ausente: {removed:,}")
print(f"✅ Registros válidos para modelagem: {len(df_model):,}")

# ------------------------------------------------------------
# 4. CONVERTER TARGET PARA 0/1
# ------------------------------------------------------------

target_normalized = (
    df_model[target_col]
    .astype(str)
    .str.strip()
    .str.lower()
)

target_map = {
    "no": 0,
    "yes": 1,
    "não": 0,
    "nao": 0,
    "sim": 1,
    "false": 0,
    "true": 1,
    "0": 0,
    "1": 1,
    "0.0": 0,
    "1.0": 1
}

y = target_normalized.map(target_map)

# ------------------------------------------------------------
# 5. VALIDAR CONVERSÃO
# ------------------------------------------------------------

if y.isna().any():

    invalid = target_normalized[y.isna()].unique()

    raise ValueError(
        f"❌ Valores de target não reconhecidos: {invalid}"
    )

y = y.astype(int)

# ------------------------------------------------------------
# 6. CRIAR FEATURES X
# ------------------------------------------------------------

# Remove TODAS as colunas relacionadas a churn
# para impedir target leakage.

churn_cols = [
    col for col in df_model.columns
    if "churn" in col.lower()
]

X = df_model.drop(columns=[*churn_cols, "customerID"]).copy()

# Resetar índices para garantir alinhamento perfeito
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

# ------------------------------------------------------------
# 7. AUDITORIA FINAL
# ------------------------------------------------------------

print("\n📊 DISTRIBUIÇÃO FINAL DO TARGET")
print("-" * 45)

counts = y.value_counts().sort_index()
pct = y.value_counts(normalize=True).sort_index() * 100

print(
    f"Não Churn (0): {counts.get(0, 0):,} "
    f"({pct.get(0, 0):.2f}%)"
)

print(
    f"Churn     (1): {counts.get(1, 0):,} "
    f"({pct.get(1, 0):.2f}%)"
)

# ------------------------------------------------------------
# 8. VERIFICAÇÕES DE INTEGRIDADE
# ------------------------------------------------------------

print("\n🔎 VERIFICAÇÕES")

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

assert len(X) == len(y), "❌ X e y possuem tamanhos diferentes."
assert y.isna().sum() == 0, "❌ Ainda existem NaNs no target."
assert set(y.unique()) == {0, 1}, "❌ Target não é binário."

leakage = [
    col for col in X.columns
    if "churn" in col.lower()
]

assert len(leakage) == 0, \
    f"❌ Possível target leakage: {leakage}"

print("✅ X e y alinhados")
print("✅ Target binário")
print("✅ Nenhum target ausente")
print("✅ Nenhuma variável de churn em X")
print("✅ Nenhum target leakage detectado")

print("\n" + "=" * 60)
print("✅ BLOCO 2 CONCLUÍDO COM SUCESSO")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCO 3 — SPLIT TREINO/TESTE + PRÉ-PROCESSAMENTO
# Telecom X BR II — Churn Prediction
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

print("🔧 BLOCO 3 — SPLIT + PRÉ-PROCESSAMENTO")
print("=" * 60)

# ------------------------------------------------------------
# 1. DIVIDIR TREINO E TESTE
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n✂️ DIVISÃO DOS DADOS")
print("-" * 45)

print(f"Treino: {X_train.shape[0]:,} registros")
print(f"Teste : {X_test.shape[0]:,} registros")

# ------------------------------------------------------------
# 2. VERIFICAR DISTRIBUIÇÃO DO TARGET
# ------------------------------------------------------------

def mostrar_distribuicao(nome, target):

    counts = target.value_counts().sort_index()
    pct = target.value_counts(normalize=True).sort_index() * 100

    print(f"\n📊 {nome}")

    print(
        f"Não Churn (0): {counts.get(0, 0):,} "
        f"({pct.get(0, 0):.2f}%)"
    )

    print(
        f"Churn     (1): {counts.get(1, 0):,} "
        f"({pct.get(1, 0):.2f}%)"
    )

mostrar_distribuicao("TREINO", y_train)
mostrar_distribuicao("TESTE", y_test)

# ------------------------------------------------------------
# 3. IDENTIFICAR TIPOS DE FEATURES
# ------------------------------------------------------------

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n🧩 TIPOS DE FEATURES")
print("-" * 45)

print(f"Numéricas   : {len(numeric_features)}")
print(f"Categóricas : {len(categorical_features)}")

# ------------------------------------------------------------
# 4. PIPELINE PARA VARIÁVEIS NUMÉRICAS
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# ------------------------------------------------------------
# 5. PIPELINE PARA VARIÁVEIS CATEGÓRICAS
# ------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=None
            )
        )
    ]
)

# ------------------------------------------------------------
# 6. CONSTRUIR PRÉ-PROCESSADOR
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

print("\n⚙️ Pré-processador criado.")

# O pipeline ajustará o pré-processador dentro de cada partição de treino.
assert X_train.index.intersection(X_test.index).empty
assert "customerID" not in X_train.columns
assert set(y_train.unique()) == set(y_test.unique()) == {0, 1}


In [ ]:
# BLOCO 4 — COMPARAR MODELOS POR VALIDAÇÃO CRUZADA NO TREINO
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC_AUC": "roc_auc"
}

results = []
for name, model in models.items():
    candidate = Pipeline([("preprocessor", preprocessor), ("model", model)])
    scores = cross_validate(candidate, X_train, y_train, cv=cv, scoring=scoring)
    results.append({"Modelo": name, **{
        metric: scores[f"test_{metric}"].mean() for metric in scoring
    }})

results_df = pd.DataFrame(results).sort_values(
    by=["Recall", "F1", "ROC_AUC"], ascending=False
).reset_index(drop=True)
display(results_df.round(4))
best_model_name = results_df.loc[0, "Modelo"]
print(f"Melhor baseline na validação cruzada: {best_model_name}")


In [ ]:
# BLOCO 5 — OTIMIZAR O MODELO SELECIONADO NO TREINO
from sklearn.model_selection import GridSearchCV

if best_model_name == "Logistic Regression":

    model_to_optimize = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    param_grid = {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["liblinear", "saga"],
        "penalty": ["l1", "l2"]
    }

elif best_model_name == "Random Forest":

    model_to_optimize = RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    )

    param_grid = {
        "n_estimators": [200, 400],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }

elif best_model_name == "Gradient Boosting":

    model_to_optimize = GradientBoostingClassifier(
        random_state=42
    )

    param_grid = {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3],
        "min_samples_leaf": [1, 3]
    }

else:
    raise ValueError(
        f"❌ Modelo não reconhecido: {best_model_name}"
    )


pipeline = Pipeline([("preprocessor", preprocessor), ("model", model_to_optimize)])
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid={f"model__{name}": values for name, values in param_grid.items()},
    scoring="recall",
    cv=cv,
    n_jobs=2,
    error_score="raise"
)
grid_search.fit(X_train, y_train)
final_model = grid_search.best_estimator_
print(f"Parâmetros selecionados: {grid_search.best_params_}")
print(f"Recall médio na validação cruzada: {grid_search.best_score_:.4f}")


In [ ]:
# ============================================================
# BLOCO 6 — AVALIAÇÃO FINAL DO MODELO
# Telecom X BR II — Churn Prediction
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

print("📊 BLOCO 6 — AVALIAÇÃO FINAL")
print("=" * 60)

# ------------------------------------------------------------
# 1. DEFINIR MODELO E DADOS CORRETOS
# ------------------------------------------------------------

model_eval = final_model

X_eval = X_test
y_eval = y_test

print(f"\n✅ Modelo: {type(model_eval.named_steps['model']).__name__}")
print(f"✅ Amostras de teste: {len(y_eval)}")
print(f"✅ Colunas de entrada: {X_eval.shape[1]}")

# ------------------------------------------------------------
# 2. PREDIÇÕES
# ------------------------------------------------------------

y_pred_final = model_eval.predict(X_eval)

y_prob_final = model_eval.predict_proba(X_eval)[:, 1]

print("✅ Predições geradas")

# ------------------------------------------------------------
# 3. MÉTRICAS
# ------------------------------------------------------------

accuracy_final = accuracy_score(
    y_eval, y_pred_final
)

precision_final = precision_score(
    y_eval, y_pred_final,
    zero_division=0
)

recall_final = recall_score(
    y_eval, y_pred_final,
    zero_division=0
)

f1_final = f1_score(
    y_eval, y_pred_final,
    zero_division=0
)

roc_auc_final = roc_auc_score(
    y_eval, y_prob_final
)

metrics_final = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Resultado": [
        accuracy_final,
        precision_final,
        recall_final,
        f1_final,
        roc_auc_final
    ]
})

print("\n📈 MÉTRICAS FINAIS")
print("-" * 50)

display(metrics_final.round(4))

# ------------------------------------------------------------
# 4. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n📋 CLASSIFICATION REPORT")
print("-" * 50)

print(
    classification_report(
        y_eval,
        y_pred_final,
        target_names=[
            "Não Churn",
            "Churn"
        ],
        zero_division=0
    )
)

# ------------------------------------------------------------
# 5. MATRIZ DE CONFUSÃO
# ------------------------------------------------------------

cm = confusion_matrix(
    y_eval,
    y_pred_final
)

tn, fp, fn, tp = cm.ravel()

print("\n🔎 MATRIZ DE CONFUSÃO")
print("-" * 50)

print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")

os.makedirs("images", exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 5))

ax.imshow(cm)

ax.set_title(
    "Matriz de Confusão — Modelo Final"
)

ax.set_xlabel("Predição")
ax.set_ylabel("Valor Real")

ax.set_xticks([0, 1])
ax.set_xticklabels([
    "Não Churn",
    "Churn"
])

ax.set_yticks([0, 1])
ax.set_yticklabels([
    "Não Churn",
    "Churn"
])

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            fontsize=14
        )

plt.tight_layout()

plt.savefig(
    "images/confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. CURVA ROC
# ------------------------------------------------------------

fpr, tpr, _ = roc_curve(
    y_eval,
    y_prob_final
)

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(
    fpr,
    tpr,
    label=f"AUC = {roc_auc_final:.3f}"
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Aleatório"
)

ax.set_title(
    "Curva ROC — Modelo Final"
)

ax.set_xlabel(
    "False Positive Rate"
)

ax.set_ylabel(
    "True Positive Rate"
)

ax.legend()

plt.tight_layout()

plt.savefig(
    "images/roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. RESUMO
# ------------------------------------------------------------

print("\n💼 RESUMO FINAL")
print("=" * 60)

print(f"Accuracy : {accuracy_final:.2%}")
print(f"Precision: {precision_final:.2%}")
print(f"Recall   : {recall_final:.2%}")
print(f"F1-score : {f1_final:.2%}")
print(f"ROC-AUC  : {roc_auc_final:.4f}")

print("\nMatriz de confusão:")
print(
    f"TN={tn} | FP={fp} | "
    f"FN={fn} | TP={tp}"
)

print("\n📁 Gráficos salvos:")
print("images/confusion_matrix.png")
print("images/roc_curve.png")

print("\n" + "=" * 60)
print("✅ BLOCO 6 CONCLUÍDO COM SUCESSO")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCO 7 — CONCLUSÃO FINAL E INSIGHTS DE NEGÓCIO
# Telecom X BR II — Churn Prediction
# ============================================================

import os
import json
import pandas as pd

print("🎯 BLOCO 7 — CONCLUSÃO FINAL E INSIGHTS DE NEGÓCIO")
print("=" * 70)

# ------------------------------------------------------------
# 1. VALIDAR RESULTADOS DO BLOCO 6
# ------------------------------------------------------------

required_vars = [
    "accuracy_final",
    "precision_final",
    "recall_final",
    "f1_final",
    "roc_auc_final",
    "tn",
    "fp",
    "fn",
    "tp"
]

missing_vars = [
    var for var in required_vars
    if var not in globals()
]

if missing_vars:
    raise NameError(
        "❌ Execute o Bloco 6 primeiro. "
        f"Variáveis ausentes: {missing_vars}"
    )

print("\n✅ Resultados do modelo carregados")

# ------------------------------------------------------------
# 2. RESUMO EXECUTIVO
# ------------------------------------------------------------

total_test = tn + fp + fn + tp
actual_churn = fn + tp
predicted_churn = fp + tp

churn_rate_test = (
    actual_churn / total_test
    if total_test > 0 else 0
)

print("\n📊 RESUMO EXECUTIVO")
print("-" * 70)

print(
    f"""
O modelo final de Machine Learning foi desenvolvido para identificar
clientes com maior probabilidade de churn na Telecom X.

Desempenho no conjunto de teste:

• Accuracy : {accuracy_final:.2%}
• Precision: {precision_final:.2%}
• Recall   : {recall_final:.2%}
• F1-score : {f1_final:.2%}
• ROC-AUC  : {roc_auc_final:.4f}

O conjunto de teste contém {total_test:,} clientes.

Churn real identificado:
{actual_churn:,} clientes ({churn_rate_test:.2%})

Clientes classificados pelo modelo como churn:
{predicted_churn:,}
"""
)

# ------------------------------------------------------------
# 3. INTERPRETAÇÃO DA MATRIZ DE CONFUSÃO
# ------------------------------------------------------------

print("\n🔎 INTERPRETAÇÃO DA MATRIZ DE CONFUSÃO")
print("-" * 70)

print(f"""
True Negatives  (TN): {tn:,}
Clientes corretamente identificados como NÃO churn.

False Positives (FP): {fp:,}
Clientes classificados como churn, embora não tenham cancelado.

False Negatives (FN): {fn:,}
Clientes que realmente apresentaram churn, mas não foram
identificados pelo modelo.

True Positives  (TP): {tp:,}
Clientes com churn corretamente identificados pelo modelo.
""")

# ------------------------------------------------------------
# 4. EFICIÊNCIA NA DETECÇÃO DE CHURN
# ------------------------------------------------------------

detected_rate = (
    tp / actual_churn
    if actual_churn > 0 else 0
)

missed_rate = (
    fn / actual_churn
    if actual_churn > 0 else 0
)

print("\n🎯 CAPACIDADE DE DETECÇÃO")
print("-" * 70)

print(
    f"O modelo identifica {detected_rate:.2%} "
    "dos casos reais de churn."
)

print(
    f"A proporção de churn não detectado é "
    f"{missed_rate:.2%}."
)

# ------------------------------------------------------------
# 5. IMPLICAÇÕES PARA O NEGÓCIO
# ------------------------------------------------------------

print("\n💼 IMPLICAÇÕES PARA O NEGÓCIO")
print("-" * 70)

print("""
A aplicação do modelo permite transformar a análise histórica
de churn em uma ferramenta de apoio à decisão.

Em um cenário operacional, os scores de probabilidade poderiam
ser utilizados para priorizar clientes em campanhas de retenção,
permitindo direcionar recursos para grupos com maior risco.

Possíveis aplicações:

1. Priorização de clientes com maior probabilidade de churn.

2. Campanhas de retenção segmentadas por nível de risco.

3. Ofertas comerciais personalizadas para clientes vulneráveis.

4. Acompanhamento de indicadores de churn e retenção.

5. Integração dos scores do modelo com CRM e dashboards.

6. Monitoramento periódico do desempenho do modelo.
""")

# ------------------------------------------------------------
# 6. RECOMENDAÇÕES ESTRATÉGICAS
# ------------------------------------------------------------

print("\n🚀 RECOMENDAÇÕES ESTRATÉGICAS")
print("-" * 70)

print("""
A estratégia de retenção não deve tratar todos os clientes
da mesma forma.

Uma abordagem recomendada seria criar três grupos:

🔴 ALTO RISCO
Contato proativo, análise individual e ofertas específicas
de retenção.

🟡 MÉDIO RISCO
Campanhas segmentadas, benefícios e acompanhamento de
engajamento.

🟢 BAIXO RISCO
Relacionamento regular e monitoramento preventivo.

O score preditivo pode funcionar como uma camada adicional
de inteligência para CRM e Customer Success.
""")

# ------------------------------------------------------------
# 7. LIMITAÇÕES
# ------------------------------------------------------------

print("\n⚠️ LIMITAÇÕES DO MODELO")
print("-" * 70)

print("""
Os resultados devem ser interpretados dentro do contexto
do dataset utilizado.

Antes de uma aplicação em produção, seriam necessárias etapas
adicionais, incluindo:

• validação com dados recentes;
• análise temporal do comportamento do churn;
• monitoramento de data drift e concept drift;
• definição do threshold de classificação com base no custo
  de falsos positivos e falsos negativos;
• análise financeira das ações de retenção;
• reavaliação periódica do modelo.
""")

# ------------------------------------------------------------
# 8. CONCLUSÃO FINAL
# ------------------------------------------------------------

print("\n🏁 CONCLUSÃO FINAL")
print("=" * 70)

print(f"""
O projeto demonstrou a construção de um pipeline de Machine
Learning para previsão de churn, passando pela preparação dos
dados, definição da variável-alvo, pré-processamento, treinamento,
comparação de modelos e avaliação final.

O modelo selecionado apresentou:

Accuracy  = {accuracy_final:.2%}
Precision = {precision_final:.2%}
Recall    = {recall_final:.2%}
F1-score  = {f1_final:.2%}
ROC-AUC   = {roc_auc_final:.4f}

Mais importante do que a acurácia isoladamente, o modelo deve
ser avaliado pela capacidade de identificar clientes com risco
de churn e pelo impacto que suas previsões podem gerar sobre
decisões de retenção.

Assim, a modelagem preditiva deixa de ser apenas um exercício
estatístico e passa a funcionar como instrumento de apoio à
tomada de decisão, permitindo transformar dados de clientes
em ações potencialmente mais direcionadas de relacionamento
e retenção.
""")

# ------------------------------------------------------------
# 9. SALVAR RESULTADOS
# ------------------------------------------------------------

os.makedirs("results", exist_ok=True)

final_results = {
    "model": type(model_eval.named_steps["model"]).__name__,
    "accuracy": float(accuracy_final),
    "precision": float(precision_final),
    "recall": float(recall_final),
    "f1_score": float(f1_final),
    "roc_auc": float(roc_auc_final),
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    },
    "test_samples": int(total_test),
    "actual_churn": int(actual_churn),
    "churn_detection_rate": float(detected_rate)
}

with open(
    "results/final_model_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_results,
        f,
        indent=4,
        ensure_ascii=False
    )

metrics_export = pd.DataFrame({
    "metric": [
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "roc_auc"
    ],
    "value": [
        accuracy_final,
        precision_final,
        recall_final,
        f1_final,
        roc_auc_final
    ]
})

metrics_export.to_csv(
    "results/final_metrics.csv",
    index=False
)

print("\n💾 RESULTADOS SALVOS")
print("-" * 70)

print("✅ results/final_model_results.json")
print("✅ results/final_metrics.csv")
print("✅ images/confusion_matrix.png")
print("✅ images/roc_curve.png")

print("\n" + "=" * 70)
print("🎉 PROJETO TELECOM X BR II FINALIZADO")
print("=" * 70)